# Rescue Drone Mission 20 — Real Bengali Audio Distress Classification
**Platform**: Kaggle NVIDIA Tesla T4 GPU Runtime (16 GB VRAM)
**Dataset**: `allfaz/rescue-drone-audio-1500` (1,500 Calibrated 16 kHz WAVs)
**Splits**: Train (900 samples), Val (300 samples), Locked Test (300 samples)
**Task**: Bengali Distress Cry ('bachao' / বাঁচাও) vs Normal Bengali Conversational Speech
**Committer**: Mohammad Sayem Uddin <sayemwork41@gmail.com>


In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


In [ ]:
!pip install -q scikit-learn pandas numpy


In [ ]:
import os, pandas as pd
from pathlib import Path
dataset_path = Path('/kaggle/input/rescue-drone-audio-1500')
print('Dataset directory exists:', dataset_path.exists())
if dataset_path.exists():
    manifest_p = dataset_path / 'manifest.csv'
    df = pd.read_csv(manifest_p)
    print(f'Total manifest rows: {len(df)}')
    print('Split counts:', dict(df['split'].value_counts()))
    pos_wavs = len(list((dataset_path / 'audio' / 'positive').glob('*.wav')))
    neg_wavs = len(list((dataset_path / 'audio' / 'negative').glob('*.wav')))
    print(f'WAV files found on disk: {pos_wavs} positive, {neg_wavs} negative')


In [ ]:
from pathlib import Path
runner_content = '#!/usr/bin/env python3\n"""Audio Emergency Vocalization Detection — Reusable Kaggle GPU Runner.\n\nTarget Environment: Kaggle Cloud, NVIDIA Tesla T4 GPU (16 GB VRAM).\nClassification Task:\n  - Class 1: Positive Emergency Distress Cry (\'bachao\' / বাঁচাও)\n  - Class 0: Negative Normal Bengali Conversational Speech & Ambient Noise\nAcoustic Features:\n  - 16 kHz Mono, 2.0s (32,000 samples)\n  - Pure PyTorch STFT + Mel Filterbank -> Log-Mel Spectrogram (64 mel bins x 126 frames, 1 channel)\nModel:\n  - Deep 2D Convolutional Neural Network with BatchNorm, ReLU, Dropout, and Linear Head.\nEvaluation:\n  - Checkpoint selection strictly on Validation split (300 samples).\n  - Decision threshold tau* calibrated on Validation split.\n  - Frozen independent evaluation on Locked Test split (300 samples, 0 root leakage).\n  - Granular breakdown across 5 noise conditions and 6 SNR tiers.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport logging\nimport os\nimport random\nimport shutil\nimport sys\nimport time\nimport wave\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\ntry:\n    from sklearn.metrics import (\n        accuracy_score,\n        precision_score,\n        recall_score,\n        f1_score,\n        roc_auc_score,\n        precision_recall_curve,\n        auc,\n        confusion_matrix,\n    )\n    HAS_SKLEARN = True\nexcept ImportError:\n    HAS_SKLEARN = False\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger("Audio-KaggleGPU")\n\nSEED = 42\n\n\ndef set_reproducibility(seed: int = SEED) -> None:\n    """Sets random seeds across all libraries for deterministic execution."""\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n    logger.info("Deterministic seed %d initialized.", seed)\n\n\ndef audit_gpu_environment() -> Dict[str, Any]:\n    """Inspects and returns the hardware runtime profile."""\n    is_cuda = torch.cuda.is_available()\n    device_name = torch.cuda.get_device_name(0) if is_cuda else "CPU"\n    total_mem_gb = (\n        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if is_cuda else 0.0\n    )\n\n    profile = {\n        "platform": "Kaggle Cloud",\n        "cuda_available": is_cuda,\n        "device_name": device_name,\n        "gpu_memory_gb": round(total_mem_gb, 2),\n        "pytorch_version": torch.__version__,\n        "cuda_version": torch.version.cuda if is_cuda else None,\n        "is_nvidia_t4": "T4" in device_name,\n        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n    }\n    logger.info("GPU Audit: %s (%.2f GB VRAM, CUDA %s)",\n                device_name, total_mem_gb, profile["cuda_version"])\n    return profile\n\n\ndef create_mel_filterbank(\n    sr: int = 16000,\n    n_fft: int = 1024,\n    n_mels: int = 64,\n    f_min: float = 0.0,\n    f_max: float = 8000.0,\n) -> torch.Tensor:\n    """Constructs triangular Mel filterbank matrix in pure PyTorch."""\n    def hz_to_mel(hz: float) -> float:\n        return float(2595.0 * np.log10(1.0 + hz / 700.0))\n\n    def mel_to_hz(mel: float) -> float:\n        return float(700.0 * (10.0 ** (mel / 2595.0) - 1.0))\n\n    mel_min = hz_to_mel(f_min)\n    mel_max = hz_to_mel(f_max)\n    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)\n    hz_points = np.array([mel_to_hz(m) for m in mel_points])\n    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)\n\n    fbank = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)\n    for m in range(1, n_mels + 1):\n        f_prev = bin_points[m - 1]\n        f_curr = bin_points[m]\n        f_next = bin_points[m + 1]\n        for k in range(f_prev, f_curr):\n            fbank[m - 1, k] = (k - f_prev) / max(1, (f_curr - f_prev))\n        for k in range(f_curr, f_next):\n            fbank[m - 1, k] = (f_next - k) / max(1, (f_next - f_curr))\n\n    return torch.from_numpy(fbank)\n\n\nclass RealAudioSpectrogramDataset(Dataset):\n    """Loads 16 kHz WAV files and computes normalized Log-Mel spectrograms."""\n\n    def __init__(\n        self,\n        manifest_csv: Path,\n        dataset_root: Path,\n        split: str = "train",\n        n_mels: int = 64,\n        target_length: int = 32000,\n    ) -> None:\n        self.split = split\n        self.target_length = target_length\n        self.dataset_root = Path(dataset_root)\n        self.n_mels = n_mels\n        self.mel_fb = create_mel_filterbank(sr=16000, n_fft=1024, n_mels=n_mels)\n        self.window = torch.hann_window(1024)\n\n        # Read manifest rows\n        self.records: List[Dict[str, Any]] = []\n        with open(manifest_csv, "r", encoding="utf-8") as f:\n            reader = csv.DictReader(f)\n            for row in reader:\n                if row.get("split") == split:\n                    self.records.append(row)\n\n        logger.info("RealAudioDataset initialized for split \'%s\': %d samples.", split, len(self.records))\n\n    def __len__(self) -> int:\n        return len(self.records)\n\n    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:\n        row = self.records[idx]\n        cls_sub = "positive" if row["class_label"] == "bachao" else "negative"\n        wav_path = self.dataset_root / "audio" / cls_sub / row["filename"]\n\n        # Read PCM\n        if not wav_path.exists():\n            raise FileNotFoundError(f"Audio file not found at {wav_path}")\n\n        with wave.open(str(wav_path), "rb") as w:\n            n_frames = w.getnframes()\n            frames = w.readframes(n_frames)\n            audio = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0\n\n        if len(audio) < self.target_length:\n            audio = np.pad(audio, (0, self.target_length - len(audio)))\n        else:\n            audio = audio[:self.target_length]\n\n        x = torch.from_numpy(audio).float()\n\n        # Compute STFT & Mel Spectrogram\n        stft = torch.stft(\n            x,\n            n_fft=1024,\n            hop_length=256,\n            win_length=1024,\n            window=self.window,\n            return_complex=True,\n        )\n        linear_spec = torch.abs(stft)  # (513, 126)\n        mel_spec = torch.matmul(self.mel_fb, linear_spec)  # (64, 126)\n        log_mel = torch.log(mel_spec + 1e-6)\n        norm_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)\n        feature_tensor = norm_mel.unsqueeze(0)  # (1, 64, 126)\n\n        label = 1.0 if row["class_label"] == "bachao" else 0.0\n        label_tensor = torch.tensor(label, dtype=torch.float32)\n\n        meta = {\n            "filename": row["filename"],\n            "class_label": row["class_label"],\n            "noise_condition": row.get("noise_condition", "unknown"),\n            "snr_db": float(row.get("snr_db", 0.0)),\n            "distance_simulation": row.get("distance_simulation", "unknown"),\n            "root_source": row.get("root_source", "unknown"),\n        }\n        return feature_tensor, label_tensor, meta\n\n\nclass AudioBaselinePyTorchCNN(nn.Module):\n    """Deep 2D Convolutional Neural Network for Spectrogram Distress Classification."""\n\n    def __init__(self, n_mels: int = 64, dropout_rate: float = 0.3) -> None:\n        super().__init__()\n        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)\n        self.bn1 = nn.BatchNorm2d(32)\n        self.pool1 = nn.MaxPool2d(2, 2)\n\n        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)\n        self.bn2 = nn.BatchNorm2d(64)\n        self.pool2 = nn.MaxPool2d(2, 2)\n\n        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)\n        self.bn3 = nn.BatchNorm2d(128)\n        self.pool3 = nn.MaxPool2d(2, 2)\n\n        self.dropout = nn.Dropout(dropout_rate)\n        # Dimensions: 64//8 = 8, 126//8 = 15 -> 128 * 8 * 15 = 15,360\n        self.fc1 = nn.Linear(128 * 8 * 15, 64)\n        self.bn_fc = nn.BatchNorm1d(64)\n        self.fc_out = nn.Linear(64, 1)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # x: (B, 1, 64, 126)\n        x = F.relu(self.bn1(self.conv1(x)))\n        x = self.pool1(x)\n\n        x = F.relu(self.bn2(self.conv2(x)))\n        x = self.pool2(x)\n\n        x = F.relu(self.bn3(self.conv3(x)))\n        x = self.pool3(x)\n\n        x = x.flatten(1)\n        x = self.dropout(F.relu(self.bn_fc(self.fc1(x))))\n        logits = self.fc_out(x)\n        return logits.squeeze(1)\n\n\ndef evaluate_audio_split(\n    model: nn.Module,\n    loader: DataLoader,\n    device: torch.device,\n    threshold: float = 0.5,\n) -> Dict[str, Any]:\n    """Computes full evaluation metrics including noise and SNR breakdowns."""\n    model.eval()\n    all_targets: List[int] = []\n    all_probs: List[float] = []\n    all_meta: List[Dict[str, Any]] = []\n\n    with torch.no_grad():\n        for features, labels, meta in loader:\n            features = features.to(device)\n            logits = model(features)\n            probs = torch.sigmoid(logits).cpu().numpy().tolist()\n            targets = labels.numpy().astype(int).tolist()\n\n            all_probs.extend(probs)\n            all_targets.extend(targets)\n\n            # Unpack metadata batch\n            batch_size = len(targets)\n            for i in range(batch_size):\n                all_meta.append({\n                    "noise_condition": meta["noise_condition"][i],\n                    "snr_db": float(meta["snr_db"][i]),\n                    "distance_simulation": meta["distance_simulation"][i],\n                })\n\n    preds = [1 if p >= threshold else 0 for p in all_probs]\n\n    acc = float(accuracy_score(all_targets, preds)) if HAS_SKLEARN else 0.0\n    prec = float(precision_score(all_targets, preds, zero_division=0)) if HAS_SKLEARN else 0.0\n    rec = float(recall_score(all_targets, preds, zero_division=0)) if HAS_SKLEARN else 0.0\n    f1 = float(f1_score(all_targets, preds, zero_division=0)) if HAS_SKLEARN else 0.0\n    roc_auc = float(roc_auc_score(all_targets, all_probs)) if HAS_SKLEARN else 0.0\n\n    cm = confusion_matrix(all_targets, preds) if HAS_SKLEARN else np.array([[0, 0], [0, 0]])\n    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])\n\n    fpr = round(fp / (fp + tn), 4) if (fp + tn) > 0 else 0.0\n    fnr = round(fn / (fn + tp), 4) if (fn + tp) > 0 else 0.0\n\n    # Noise Breakdown\n    noise_stats: Dict[str, Dict[str, Any]] = {}\n    for noise in ["clean_speech", "drone_propeller", "flood_water", "heavy_rain", "strong_wind"]:\n        indices = [i for i, m in enumerate(all_meta) if m["noise_condition"] == noise]\n        if indices:\n            sub_t = [all_targets[i] for i in indices]\n            sub_p = [preds[i] for i in indices]\n            sub_acc = float(accuracy_score(sub_t, sub_p)) if HAS_SKLEARN else 0.0\n            sub_f1 = float(f1_score(sub_t, sub_p, zero_division=0)) if HAS_SKLEARN else 0.0\n            noise_stats[noise] = {\n                "samples": len(indices),\n                "accuracy": round(sub_acc, 4),\n                "f1": round(sub_f1, 4),\n            }\n\n    # SNR Breakdown\n    snr_stats: Dict[str, Dict[str, Any]] = {}\n    for snr in sorted(list(set(m["snr_db"] for m in all_meta))):\n        indices = [i for i, m in enumerate(all_meta) if m["snr_db"] == snr]\n        if indices:\n            sub_t = [all_targets[i] for i in indices]\n            sub_p = [preds[i] for i in indices]\n            sub_acc = float(accuracy_score(sub_t, sub_p)) if HAS_SKLEARN else 0.0\n            sub_f1 = float(f1_score(sub_t, sub_p, zero_division=0)) if HAS_SKLEARN else 0.0\n            snr_stats[str(snr)] = {\n                "samples": len(indices),\n                "accuracy": round(sub_acc, 4),\n                "f1": round(sub_f1, 4),\n            }\n\n    return {\n        "threshold": threshold,\n        "accuracy": round(acc, 4),\n        "precision": round(prec, 4),\n        "recall": round(rec, 4),\n        "f1": round(f1, 4),\n        "roc_auc": round(roc_auc, 4),\n        "fpr": fpr,\n        "fnr": fnr,\n        "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},\n        "noise_breakdown": noise_stats,\n        "snr_breakdown": snr_stats,\n        "all_probs": all_probs,\n        "all_targets": all_targets,\n    }\n\n\ndef train_real_audio_classifier(\n    dataset_dir: str = "/kaggle/input/rescue-drone-audio-1500",\n    epochs: int = 15,\n    batch_size: int = 32,\n    lr: float = 0.001,\n    output_dir: str = "/kaggle/working/artifacts/audio_gpu",\n) -> Dict[str, Any]:\n    """Orchestrates end-to-end real audio dataset training and evaluation on Kaggle GPU."""\n    set_reproducibility(SEED)\n    gpu_profile = audit_gpu_environment()\n\n    out_p = Path(output_dir)\n    out_p.mkdir(parents=True, exist_ok=True)\n\n    data_p = Path(dataset_dir)\n    manifest_p = data_p / "manifest.csv"\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    logger.info("Initializing AudioBaselinePyTorchCNN on device: %s", device)\n\n    train_ds = RealAudioSpectrogramDataset(manifest_p, data_p, split="train")\n    val_ds = RealAudioSpectrogramDataset(manifest_p, data_p, split="val")\n    test_ds = RealAudioSpectrogramDataset(manifest_p, data_p, split="test")\n\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)\n    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)\n\n    model = AudioBaselinePyTorchCNN().to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    criterion = nn.BCEWithLogitsLoss()\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n\n    history: List[Dict[str, Any]] = []\n    best_val_f1 = -1.0\n    best_weights_path = out_p / "audio_best_weights.pt"\n\n    start_time = time.perf_counter()\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        train_loss = 0.0\n        train_correct = 0\n        total_train = 0\n\n        for features, labels, _ in train_loader:\n            features = features.to(device)\n            labels = labels.to(device)\n\n            optimizer.zero_grad()\n            logits = model(features)\n            loss = criterion(logits, labels)\n            loss.backward()\n            optimizer.step()\n\n            train_loss += loss.item() * len(labels)\n            preds = (torch.sigmoid(logits) >= 0.5).float()\n            train_correct += (preds == labels).sum().item()\n            total_train += len(labels)\n\n        scheduler.step()\n        train_loss /= total_train\n        train_acc = train_correct / total_train\n\n        # Validation evaluation\n        val_eval = evaluate_audio_split(model, val_loader, device, threshold=0.5)\n\n        epoch_record = {\n            "epoch": epoch,\n            "train_loss": round(train_loss, 4),\n            "train_accuracy": round(train_acc, 4),\n            "val_accuracy": val_eval["accuracy"],\n            "val_f1": val_eval["f1"],\n            "val_roc_auc": val_eval["roc_auc"],\n        }\n        history.append(epoch_record)\n        logger.info("Epoch %2d/%2d - Loss: %.4f - TrainAcc: %.4f - ValAcc: %.4f - ValF1: %.4f - ValAUC: %.4f",\n                    epoch, epochs, train_loss, train_acc, val_eval["accuracy"], val_eval["f1"], val_eval["roc_auc"])\n\n        if val_eval["f1"] > best_val_f1:\n            best_val_f1 = val_eval["f1"]\n            torch.save(model.state_dict(), best_weights_path)\n            logger.info("  --> Saved new best checkpoint with Val F1: %.4f", best_val_f1)\n\n    elapsed_s = time.perf_counter() - start_time\n    logger.info("Training complete in %.2f seconds.", elapsed_s)\n\n    # Load best checkpoint\n    model.load_state_dict(torch.load(best_weights_path, map_location=device))\n\n    # Calibrate decision threshold on validation set\n    val_preds_dict = evaluate_audio_split(model, val_loader, device, threshold=0.5)\n    best_tau = 0.5\n    best_tau_f1 = -1.0\n    for tau in np.arange(0.1, 0.92, 0.02):\n        tau = round(float(tau), 2)\n        eval_t = evaluate_audio_split(model, val_loader, device, threshold=tau)\n        if eval_t["f1"] > best_tau_f1:\n            best_tau_f1 = eval_t["f1"]\n            best_tau = tau\n\n    logger.info("Calibrated optimal decision threshold on Validation: tau* = %.2f (Val F1 = %.4f)",\n                best_tau, best_tau_f1)\n\n    # Frozen Independent Evaluation on Locked Test Split\n    test_eval = evaluate_audio_split(model, test_loader, device, threshold=best_tau)\n\n    # Strip large raw arrays from JSON export\n    del test_eval["all_probs"]\n    del test_eval["all_targets"]\n\n    summary = {\n        "experiment_id": "audio_gpu_real_distress_benchmark",\n        "dataset_dir": dataset_dir,\n        "gpu_profile": gpu_profile,\n        "training": {\n            "epochs": epochs,\n            "batch_size": batch_size,\n            "learning_rate": lr,\n            "training_time_seconds": round(elapsed_s, 2),\n            "best_checkpoint": str(best_weights_path),\n            "calibrated_threshold": best_tau,\n        },\n        "training_history": history,\n        "locked_test_metrics": test_eval,\n    }\n\n    out_json = out_p / "mission20_audio_gpu_results.json"\n    out_json.write_text(json.dumps(summary, indent=2))\n    logger.info("Saved complete audio benchmark results to %s", out_json)\n\n    # Save results CSV\n    with open(out_p / "audio_results.csv", "w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "train_accuracy", "val_accuracy", "val_f1", "val_roc_auc"])\n        writer.writeheader()\n        writer.writerows(history)\n\n    return summary\n\n\nif __name__ == "__main__":\n    profile = audit_gpu_environment()\n    print(json.dumps(profile, indent=2))\n'
Path('/kaggle/working/audio_gpu_runner.py').write_text(runner_content)
print('Runner written to /kaggle/working/audio_gpu_runner.py')


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')
from audio_gpu_runner import train_real_audio_classifier

# Execute end-to-end real audio dataset training and independent test evaluation
audio_results = train_real_audio_classifier(
    dataset_dir='/kaggle/input/rescue-drone-audio-1500',
    epochs=15,
    batch_size=32,
    lr=0.001,
    output_dir='/kaggle/working/artifacts/audio_gpu',
)

print('--- AUDIO BENCHMARK COMPLETE ---')
print('Calibrated Decision Threshold (tau*):', audio_results['training']['calibrated_threshold'])
print('Locked Test Metrics:', audio_results['locked_test_metrics'])
